<a href="https://colab.research.google.com/github/k329272/ASLite-core/blob/main/train/gloss.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub

thedevastator_unlock_the_power_of_english_asl_with_aslg_pc12_c_path = (
    kagglehub.dataset_download(
        "thedevastator/unlock-the-power-of-english-asl-with-aslg-pc12-c"
    )
)
christianahayomide_using_assembyai_path = kagglehub.notebook_output_download(
    "christianahayomide/using-assembyai"
)

print("Data source import complete.")

100%|██████████| 4.38M/4.38M [00:01<00:00, 3.16MB/s]

Extracting files...


Extracting files...
Data source import complete.


In [2]:
!pip install transformers peft datasets accelerate evaluate -q

import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)
import torch, os

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.6 MB/s eta 0:00:00


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
# data loading
# data = pd.read_csv('/kaggle/input/unlock-the-power-of-english-asl-with-aslg-pc12-c/train.csv')
data = pd.read_csv(
    "/root/.cache/kagglehub/datasets/thedevastator/unlock-the-power-of-english-asl-with-aslg-pc12-c/versions/2/train.csv"
)
data.head()

# drop any missing value
data = data[["gloss", "text"]].dropna()

data.head()
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 87710 entries, 0 to 87709
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   gloss   87710 non-null  object
 1   text    87710 non-null  object
dtypes: object(2)
memory usage: 1.3+ MB


In [5]:
# load the test data
# test_df = pd.read_csv('/kaggle/input/using-assembyai/predicted_textx')
test_df = pd.read_csv(
    "/root/.cache/kagglehub/notebooks/christianahayomide/using-assembyai/output/versions/7/predicted_textx"
)
test_df.rename(columns={"Text": "text"}, inplace=True)
test_df.head()

,text
0,its images were used by among other palestinia...
1,the argand lamp used whale oil olive oil and o...
2,alice was default reading to the point where i...
3,the sanquala mountain ranges were first explor...
4,a slough is a wetland usually a swamp or shall...


In [6]:
# split into trainig and evaluation set
train_df, eval_df = train_test_split(data, test_size=0.3)

# convert df to csv
train_df.to_csv("train.csv", index=False)
eval_df.to_csv("eval.csv", index=False)
# test_df.to_csv("test.csv", index=False)

In [7]:
# load Dataset into HuggingFace
train_dataset = load_dataset("csv", data_files="train.csv")["train"]
eval_dataset = load_dataset("csv", data_files="eval.csv")["train"]

test_df = test_df.dropna(subset=["text"]).reset_index(drop=True)

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [8]:
# load pretrained model (t5-small)
model_name = "t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name, device_map="auto").to(
    "cuda"
)  # , torch_dtype=torch.float16
# model = model.to(device)

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [9]:
# preprocess data
def preprocess(examples):
    inputs = [g for g in examples["gloss"]]
    targets = [t for t in examples["text"]]

    model_inputs = tokenizer(
        inputs, max_length=64, truncation=True, padding="max_length"
    )

    labels = tokenizer(
        text_target=targets, max_length=64, truncation=True, padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


train_dataset = train_dataset.map(preprocess, batched=True)
eval_dataset = eval_dataset.map(preprocess, batched=True)

Map:   0%|          | 0/61397 [00:00<?, ? examples/s]

Map:   0%|          | 0/26313 [00:00<?, ? examples/s]

In [10]:
import os

os.environ["WANDB_DISABLED"] = "true"

In [11]:
# training setup
training_args = Seq2SeqTrainingArguments(
    output_dir="/kaggle/working/",
    eval_strategy="epoch",
    learning_rate=0.00005,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=12,
    weight_decay=0.01,
    save_total_limit=2,
    predict_with_generate=True,
    logging_dir="/kaggle/working/logs",
    logging_steps=100,
    push_to_hub=False,
    report_to="none",
    fp16=False,
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [12]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    # label_smoothing_factor = 0.1
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,0.211479,0.133011
2,0.139447,0.102088
3,0.121833,0.088518
4,0.112943,0.080715
5,0.104009,0.075122
6,0.084733,0.071913
7,0.086147,0.069553
8,0.084722,0.067683
9,0.079324,0.066156
10,0.076419,0.065624


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=92100, training_loss=0.11772478664866229, metrics={'train_runtime': 12729.4486, 'train_samples_per_second': 57.879, 'train_steps_per_second': 7.235, 'total_flos': 1.2464370877464576e+16, 'train_loss': 0.11772478664866229, 'epoch': 12.0})

In [15]:
import torch.nn.utils.prune as prune
import zipfile
from google.colab import files
import os

test_samples = test_df.head(10).copy()
display(test_samples[["text"]])


quantized_model = torch.quantization.quantize_dynamic(
    model.to("cpu"), {torch.nn.Linear}, dtype=torch.qint8
)

for name, module in quantized_model.named_modules():
    if "encoder" in name and isinstance(module, torch.nn.Linear):
        prune.l1_unstructured(module, name="weight", amount=0.2)
        prune.remove(module, "weight")

optimized_model_path = "./optimized_t5_model"
os.makedirs(optimized_model_path, exist_ok=True)

# Save the quantized model's state dictionary
torch.save(quantized_model.state_dict(), os.path.join(optimized_model_path, "pytorch_model.bin"))

# Save the original model's configuration (needed for loading with transformers)
model.config.save_pretrained(optimized_model_path)

# Save the tokenizer
tokenizer.save_pretrained(optimized_model_path)


def zip_folder(folder_path, output_path):
    with zipfile.ZipFile(output_path, "w", zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files_in_dir in os.walk(folder_path):
            for file in files_in_dir:
                zipf.write(
                    os.path.join(root, file),
                    os.path.relpath(os.path.join(root, file), folder_path),
                )


zip_folder(optimized_model_path, "model_export.zip")

files.download("model_export.zip")

,text
0,its images were used by among other palestinia...
1,the argand lamp used whale oil olive oil and o...
2,alice was default reading to the point where i...
3,the sanquala mountain ranges were first explor...
4,a slough is a wetland usually a swamp or shall...
5,vince shared a location with you
6,oron people encountered western education rela...
7,the chinese classification of mental disorders...
8,one can also define paths and loops in pointed...
9,closing the google assistant app prevents it f...


/tmp/ipykernel_1654/2840965530.py:10: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = torch.quantization.quantize_dynamic(


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>